In [1]:
%matplotlib inline
# Cell 1 — parameters (match step 1)
LAT       = 16.8167
LON       = -2.9833
RADIUS_KM = 100
LEVEL     = '06'
EPSILON   = 0.001

In [2]:
# Cell 2 — imports and connection
import sys
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '../../../..')
from scripts.shared.db_utils import db_connect
import scripts.shared.db_utils as _dbu

conn = db_connect()
print('Connected:', conn.execute('SELECT current_database()').fetchone())

ROOT  = Path(_dbu.__file__).parent.parent.parent
OUT   = ROOT / 'output' / 'edop' / 'areas'
OUT.mkdir(parents=True, exist_ok=True)

TABLE = f'public.basin{LEVEL}'
VIEW  = f'public.v_basin{LEVEL}_persist_rev1'

Connected: ('cedop',)


In [3]:
# Cell 3 — re-run buffer resolver → weighted basin set + hybas_id list
RADIUS_M = RADIUS_KM * 1000

resolver_sql = f"""
WITH pt AS (
    SELECT ST_SetSRID(ST_MakePoint({LON}, {LAT}), 4326)::geography AS pt_geog
),
buf AS (
    SELECT ST_Buffer(pt_geog, {RADIUS_M}) AS buf_geog,
           ST_Area(ST_Buffer(pt_geog, {RADIUS_M})) AS buf_area_m2
    FROM pt
),
candidates AS (
    SELECT b.hybas_id,
           ST_Area(ST_Intersection(b.geog, buf.buf_geog)) AS overlap_m2,
           buf.buf_area_m2
    FROM {TABLE} b, buf
    WHERE ST_Intersects(b.geog, buf.buf_geog)
)
SELECT hybas_id, overlap_m2 / buf_area_m2 AS weight
FROM candidates
WHERE overlap_m2 / buf_area_m2 >= {EPSILON}
ORDER BY weight DESC
"""

basin_set  = pd.read_sql(resolver_sql, conn).set_index('hybas_id')
hybas_ids  = basin_set.index.tolist()
ids_clause = ', '.join(str(h) for h in hybas_ids)

print(f'Basin set: {len(hybas_ids)} basins')
print(basin_set.round(4))

Basin set: 9 basins
              weight
hybas_id            
1.060042e+09  0.2771
1.060565e+09  0.1738
1.060041e+09  0.1628
1.060565e+09  0.1367
1.060552e+09  0.1060
1.060552e+09  0.0876
1.060044e+09  0.0255
1.060583e+09  0.0181
1.060583e+09  0.0124


In [4]:
# Cell 4 — load catalog; build variable lists and metadata DataFrame
#
# position_method column records the intended compute recipe (from CHAR Phase 5).
# Scores are not stored; they are computed on the fly below using those recipes.
# Excluded: 'deferred' (Band T — time-dependent), 'dominance_class' (pnv_shares composite).

CAT_PATH = ROOT / 'documentation' / 'EDOPS_variable_catalog_v0.3.tsv'
cat = pd.read_csv(CAT_PATH, sep='\t')

impl = cat[
    (cat['status'] == 'implemented') &
    (~cat['position_method'].isin(['deferred', 'dominance_class']))
].copy()

# Drop composite column patterns (pnv_shares: basin08_col_s = 'pnv_pc_s01..s15')
impl = impl[~impl['basin08_col_s'].fillna('').str.contains('\\.\\.')]

def _val(v):
    """Return stripped string if v is a non-null, non-empty, non-'nan' value; else None."""
    if v is None:
        return None
    try:
        if np.isnan(float(v)):
            return None
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return s if s and s.lower() != 'nan' else None

# Build flat variable list: one entry per (schema_key, s/u variant) where both
# the api_key AND the db column name are present in the catalog.
var_rows = []
for _, row in impl.iterrows():
    for su, api_col, db_col in [
        ('s', 'api_key_s', 'basin08_col_s'),
        ('u', 'api_key_u', 'basin08_col_u'),
    ]:
        api_key    = _val(row.get(api_col))
        db_colname = _val(row.get(db_col))
        if not api_key or not db_colname:
            continue
        var_rows.append({
            'api_key':         api_key,
            'schema_key':      row['schema_key'],
            'su':              su,
            'db_col':          db_colname,
            'kind':            'categorical' if row['position_method'] == 'rarity_rank' else 'continuous',
            'band':            row['band'],
            'position_method': row['position_method'],
            'typology_cluster':row.get('typology_cluster'),
        })

meta_df = pd.DataFrame(var_rows).set_index('api_key')

cont = meta_df[meta_df['kind'] == 'continuous']
cats = meta_df[meta_df['kind'] == 'categorical']

print(f'Implemented variables (excl. deferred/composite): {len(impl)} schema rows')
print(f'  continuous : {len(cont)} api_key entries')
print(f'  categorical: {len(cats)} api_key entries')
print()
print(meta_df[['schema_key','su','kind','band','position_method','typology_cluster']].to_string())

Implemented variables (excl. deferred/composite): 46 schema rows
  continuous : 43 api_key entries
  categorical: 13 api_key entries

                                             schema_key su         kind band position_method      typology_cluster
api_key                                                                                                           
elev_min                                  elevation_min  s   continuous    A      percentile  continental-gradient
elev_max                                  elevation_max  s   continuous    A      percentile       scale-dependent
slope_avg                                     slope_deg  s   continuous    A      percentile       scale-dependent
slope_upstream                                slope_deg  u   continuous    A      percentile       scale-dependent
stream_gradient                         stream_gradient  s   continuous    A      percentile       scale-dependent
lith_class                               lithology_name  s  c

In [5]:
# Cell 5 — raw values from view for our basin set
#
# SELECT * then drop geometry; view handles temperature ÷10 and lookup joins.
# Mask NoData sentinel (-9999) to NaN after fetching.

raw_sql = f"""
SELECT * FROM {VIEW}
WHERE hybas_id IN ({ids_clause})
"""

raw_all = pd.read_sql(raw_sql, conn).set_index('hybas_id')
raw_all = raw_all.drop(columns=['geom'], errors='ignore')

# Replace -9999 NoData sentinel with NaN across all numeric columns
num_cols = raw_all.select_dtypes(include='number').columns
raw_all[num_cols] = raw_all[num_cols].replace(-9999, np.nan)

# Attach weights
raw_all = basin_set[['weight']].join(raw_all)

# Subset to just our variable api_keys + weight
keep_cols = ['weight'] + [k for k in meta_df.index if k in raw_all.columns]
missing_from_view = [k for k in meta_df.index if k not in raw_all.columns]

raw_df = raw_all[keep_cols].copy()

print(f'Raw values: {len(raw_df)} basins × {len(raw_df.columns)-1} variable columns')
if missing_from_view:
    print(f'NOT found in view ({len(missing_from_view)}): {missing_from_view}')
print()
display(raw_df)

,weight,elev_min,elev_max,slope_avg,slope_upstream,stream_gradient,lith_class,karst,karst_upstream,permafrost_extent,...,pasture_extent,pasture_extent_upstream,pop_density,human_footprint_09,human_footprint_09_upstream,gdp_avg,human_dev_idx,dist_sink,endorheic,coast_flag
hybas_id,,,,,,,,,,,,,,,,,,,,,
1.060042e+09,0.277141,237,492,5,3,5,Unconsolidated Sediments (SU),0,0,0,...,0,0,1.895,15,18,2285,0.442,0.0,2,0
1.060565e+09,0.173836,256,339,2,13,5,Unconsolidated Sediments (SU),0,1,0,...,60,39,18.700,65,88,2285,0.442,2530.7,0,0
1.060041e+09,0.162803,252,325,6,6,4,Unconsolidated Sediments (SU),0,0,0,...,0,0,0.396,10,10,2285,0.442,0.0,2,0
1.060565e+09,0.136735,241,517,5,5,12,Unconsolidated Sediments (SU),0,0,0,...,37,38,8.505,49,49,2285,0.442,2530.8,0,0
1.060552e+09,0.105959,257,279,1,12,4,Unconsolidated Sediments (SU),0,1,0,...,28,39,26.705,63,86,2285,0.442,2331.1,0,0
1.060552e+09,0.087555,253,403,5,4,11,Unconsolidated Sediments (SU),0,0,0,...,50,50,3.141,50,50,2285,0.442,2331.3,0,0
1.060044e+09,0.025482,250,407,2,2,5,Unconsolidated Sediments (SU),0,0,0,...,6,6,0.430,14,14,2285,0.442,0.0,2,0
1.060583e+09,0.018053,257,1019,9,9,16,Unconsolidated Sediments (SU),0,0,0,...,48,48,15.876,64,64,2285,0.442,2625.6,0,0
1.060583e+09,0.012435,256,440,4,13,8,Unconsolidated Sediments (SU),0,1,0,...,41,39,32.727,81,89,2285,0.442,2625.8,0,0


In [6]:
# Cell 6 — position scores for continuous variables via PERCENT_RANK()
#
# One SQL scans the full basin table once, computing PERCENT_RANK() for every
# continuous variable, then filters to our hybas_ids.
# Recipes from catalog position_method:
#   percentile     → PERCENT_RANK() OVER (ORDER BY raw_val NULLS LAST)
#   log_percentile → PERCENT_RANK() OVER (ORDER BY LN(1 + GREATEST(0, raw_val)) NULLS LAST)
# NoData (-9999) and NULL → outer CASE returns NULL (not a high rank).
# Result is 0–100 (PERCENT_RANK × 100).

def rank_expr(db_col, method):
    guard = f"({db_col} = -9999 OR {db_col} IS NULL)"
    if method == 'log_percentile':
        val = f"LN(1.0 + GREATEST(0.0, {db_col}::float))"
    else:
        val = f"{db_col}::float"
    order_expr = f"CASE WHEN {guard} THEN NULL ELSE {val} END"
    prank = f"PERCENT_RANK() OVER (ORDER BY {order_expr} NULLS LAST) * 100"
    # Outer guard: if the raw value is NoData, return NULL rather than a high rank
    return f"CASE WHEN {guard} THEN NULL ELSE {prank} END"

cont_vars = meta_df[meta_df['kind'] == 'continuous']

select_parts = ['hybas_id']
alias_map    = {}

for api_key, row in cont_vars.iterrows():
    alias = f'pos_{api_key}'
    select_parts.append(f"{rank_expr(row['db_col'], row['position_method'])} AS {alias}")
    alias_map[alias] = api_key

rank_sql = f"""
WITH ranked AS (
    SELECT {', '.join(select_parts)}
    FROM {TABLE}
)
SELECT * FROM ranked
WHERE hybas_id IN ({ids_clause})
"""

pos_raw = pd.read_sql(rank_sql, conn).set_index('hybas_id')
pos_df  = pos_raw.rename(columns=alias_map)

print(f'Position scores: {len(pos_df)} basins × {len(pos_df.columns)} continuous variables')
print(f'Score range across non-null cells: {pos_df.min().min():.1f} – {pos_df.max().max():.1f}')
null_count = pos_df.isna().sum().sum()
print(f'Null scores (NoData raw values): {null_count}')
display(pos_df.round(1))

,elev_min,elev_max,slope_avg,slope_upstream,stream_gradient,karst,karst_upstream,permafrost_extent,discharge_yr,discharge_min,...,cropland_extent,cropland_extent_upstream,pasture_extent,pasture_extent_upstream,pop_density,human_footprint_09,human_footprint_09_upstream,gdp_avg,human_dev_idx,dist_sink
hybas_id,,,,,,,,,,,,,,,,,,,,,
1.060042e+09,62.0,31.8,15.4,6.7,3.7,0.0,0.0,0.0,46.3,34.9,...,0.0,0.0,0.0,0.0,41.0,26.5,27.3,9.5,5.7,0.0
1.060565e+09,62.4,33.2,15.4,12.3,12.2,0.0,0.0,0.0,54.8,47.8,...,42.9,46.6,75.7,75.5,56.5,49.5,48.5,9.5,5.7,89.1
1.060044e+09,63.4,25.9,6.2,3.9,3.7,0.0,0.0,0.0,24.5,0.0,...,0.0,0.0,48.0,45.2,28.2,25.7,24.9,9.5,5.7,0.0
1.060041e+09,63.6,19.7,18.6,14.8,2.4,0.0,0.0,0.0,18.0,0.0,...,0.0,0.0,0.0,0.0,27.6,22.3,21.7,9.5,5.7,0.0
1.060552e+09,63.8,25.6,15.4,9.1,10.9,0.0,0.0,0.0,33.4,33.6,...,42.9,40.6,82.9,83.4,46.0,50.3,49.3,9.5,5.7,86.8
1.060565e+09,64.1,20.7,6.2,30.3,3.7,0.0,61.2,0.0,86.0,89.9,...,56.4,67.7,88.0,76.1,66.5,60.1,73.6,9.5,5.7,89.1
1.060583e+09,64.1,28.2,12.3,30.3,7.3,0.0,61.2,0.0,83.8,0.0,...,59.0,67.7,78.1,76.1,73.8,69.3,74.1,9.5,5.7,90.0
1.060583e+09,64.3,55.4,26.0,21.8,17.9,0.0,0.0,0.0,69.2,59.8,...,69.7,69.4,81.8,82.1,64.5,59.5,59.2,9.5,5.7,90.0
1.060552e+09,64.3,16.3,2.8,28.2,2.4,0.0,61.2,0.0,84.9,89.3,...,0.0,66.1,70.1,76.1,71.4,58.9,72.5,9.5,5.7,86.8


In [7]:
# Cell 7 — rarity ranks for categorical variables
#
# Rarity rank = global frequency (%) of this basin's class among all non-null basins.
# Low % = rare class; high % = common class.
#
# raw_df carries text labels for some categoricals (the view joins lookup tables).
# Those labels don't match the integer class codes in the raw basin TABLE, so we
# re-fetch basin class values from the TABLE using the integer db_col, then join
# against global frequencies computed from that same integer column. Both sides
# are cast ::text for consistency, avoiding float "2.0" vs integer "2" mismatches.

cat_vars    = meta_df[meta_df['kind'] == 'categorical']
rarity_rows = {}

for api_key, row in cat_vars.iterrows():
    db_col = row['db_col']

    # Basin class values from the raw table (integer IDs via db_col, not view labels)
    basin_cls_sql = f"""
        SELECT hybas_id, {db_col}::text AS class_val
        FROM {TABLE}
        WHERE hybas_id IN ({ids_clause})
    """
    basin_cls = pd.read_sql(basin_cls_sql, conn).set_index('hybas_id')['class_val']

    # Global class frequencies — same representation as basin_cls
    freq_sql = f"""
        SELECT {db_col}::text AS class_val,
               COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS rarity_pct
        FROM {TABLE}
        WHERE {db_col} IS NOT NULL
        GROUP BY {db_col}
    """
    freq_df = pd.read_sql(freq_sql, conn).set_index('class_val')

    rarity_rows[api_key] = basin_cls.map(freq_df['rarity_pct'])

rarity_df = pd.DataFrame(rarity_rows, index=raw_df.index)

print(f'Rarity ranks: {len(rarity_df)} basins × {len(rarity_df.columns)} categorical variables')
still_null = rarity_df.columns[rarity_df.isna().all()].tolist()
if still_null:
    print(f'WARNING — still all-NaN: {still_null}')
display(rarity_df.round(2))

,lith_class,wetland_class,zone_name,strata_code,biome,eco_id,ecoregion,pnv_majority,freshwater_ecoregion_class,freshwater_ecoregion_name,land_cover_name,endorheic,coast_flag
hybas_id,,,,,,,,,,,,,
1.060042e+09,30.26,14.05,12.25,1.40,16.21,2.96,2.96,9.11,23.43,3.04,14.49,10.28,87.23
1.060565e+09,30.26,12.45,12.25,1.88,0.91,0.03,0.03,10.34,11.04,0.10,8.73,80.14,87.23
1.060041e+09,30.26,14.05,12.25,1.40,16.21,2.96,2.96,11.14,23.43,3.04,14.49,10.28,87.23
1.060565e+09,30.26,12.45,12.25,1.40,16.21,2.96,2.96,10.34,11.04,0.10,10.86,80.14,87.23
1.060552e+09,30.26,12.45,12.25,1.88,0.91,0.03,0.03,9.11,11.04,0.10,10.86,80.14,87.23
1.060552e+09,30.26,3.01,12.25,1.88,16.21,2.96,2.96,10.34,11.04,0.96,8.73,80.14,87.23
1.060044e+09,30.26,16.00,12.25,1.40,16.21,2.96,2.96,9.11,11.04,0.10,14.49,10.28,87.23
1.060583e+09,30.26,12.45,12.25,1.88,16.21,2.96,2.96,10.34,11.04,0.10,8.73,80.14,87.23
1.060583e+09,30.26,12.45,12.25,2.81,0.91,0.03,0.03,9.11,11.04,0.10,10.86,80.14,87.23


In [8]:
# Cell 8 — assemble matrix and sanity report
#
# Three outputs:
#   raw_df   — index=hybas_id; columns=api_key; values=raw values (plus 'weight')
#   score_df — index=hybas_id; columns=api_key; values=position score 0–100
#               continuous: global PERCENT_RANK; categorical: global rarity %
#   meta_df  — index=api_key; columns=schema_key, kind, band, position_method, typology_cluster
#
# These three DataFrames are the Step 2 output consumed by Step 3 (aggregation).

# Merge continuous position scores + categorical rarity ranks into one score frame
score_df = pd.concat([pos_df, rarity_df], axis=1)

# Align columns to those actually present in raw_df (excluding 'weight')
var_cols    = [c for c in raw_df.columns if c != 'weight']
score_df    = score_df.reindex(columns=var_cols)

n_basins = len(raw_df)
n_vars   = len(var_cols)

# Null audit
raw_nulls   = raw_df[var_cols].isna().sum()
score_nulls = score_df.isna().sum()
null_vars   = raw_nulls[raw_nulls > 0]

print(f'Matrix: {n_basins} basins × {n_vars} variables')
print(f'Total cells : {n_basins * n_vars}')
print(f'Null raw    : {raw_nulls.sum()}  ({raw_nulls.sum() / (n_basins * n_vars) * 100:.1f}%)')
print(f'Null scores : {score_nulls.sum()} ({score_nulls.sum() / (n_basins * n_vars) * 100:.1f}%)')

if len(null_vars):
    print(f'\nVariables with null raw values:')
    print(null_vars.to_string())

# Score range sanity
cont_scores = score_df[[k for k in var_cols if k in pos_df.columns]]
if not cont_scores.empty:
    print(f'\nContinuous score range: {cont_scores.min().min():.1f} – {cont_scores.max().max():.1f}  (expect 0–100)')
cat_scores = score_df[[k for k in var_cols if k in rarity_df.columns]]
if not cat_scores.empty:
    print(f'Rarity % range:         {cat_scores.min().min():.2f} – {cat_scores.max().max():.2f}  (expect 0–100)')

print()
print('--- raw_df (weight + first 6 continuous vars) ---')
display(raw_df[['weight'] + [c for c in var_cols[:6]]].round(3))

print('--- score_df (first 6 continuous vars) ---')
display(score_df[[c for c in var_cols[:6]]].round(1))

print('--- meta_df ---')
display(meta_df[meta_df.index.isin(var_cols)][['schema_key','su','kind','band','position_method','typology_cluster']])

,schema_key,su,kind,band,position_method,typology_cluster
api_key,,,,,,
elev_min,elevation_min,s,continuous,A,percentile,continental-gradient
elev_max,elevation_max,s,continuous,A,percentile,scale-dependent
slope_avg,slope_deg,s,continuous,A,percentile,scale-dependent
slope_upstream,slope_deg,u,continuous,A,percentile,scale-dependent
stream_gradient,stream_gradient,s,continuous,A,percentile,scale-dependent
lith_class,lithology_name,s,categorical,A,rarity_rank,NaN
karst,karst_pct,s,continuous,A,percentile,scale-dependent
karst_upstream,karst_pct,u,continuous,A,percentile,scale-dependent
permafrost_extent,permafrost_pct,s,continuous,C,percentile,continental-gradient


In [9]:
# Cell 9 — persist Step 2 outputs to output/edop/areas/
raw_df.to_csv(OUT / 'step2_raw.tsv', sep='\t', float_format='%.4f')
score_df.to_csv(OUT / 'step2_scores.tsv', sep='\t', float_format='%.2f')
meta_df.to_csv(OUT / 'step2_meta.tsv', sep='\t')

print(f'Saved to {OUT}:')
print(f'  step2_raw.tsv    ({raw_df.shape[0]} basins × {raw_df.shape[1]} cols)')
print(f'  step2_scores.tsv ({score_df.shape[0]} basins × {score_df.shape[1]} cols)')
print(f'  step2_meta.tsv   ({meta_df.shape[0]} variables)')

Saved to /Users/karlg/Documents/repos/_edops/output/edop/areas:
  step2_raw.tsv    (9 basins × 57 cols)
  step2_scores.tsv (9 basins × 56 cols)
  step2_meta.tsv   (56 variables)


In [10]:
# Cell 9 — close connection
conn.close()
print('Done.')

Done.
